# 미세조직 실습

**Microstructure · 결정립 · Grain**

결정립, 상 분포, 결함 등 현미경 규모의 조직 구조. 물성에 직접 영향을 준다.

소재 분야에서 이해하기: 결정립 크기 분포를 이미지에서 측정해 강도와 연결한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [Materials Project 용어집](https://docs.materialsproject.org/frequently-asked-questions/glossary-of-terms)

## 1. 합성 미세조직 만들기

보로노이 분할로 결정립 구조를 만들고, 이미지에서 결정립 크기를 측정합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

from scipy.spatial import cKDTree
from scipy import ndimage

size, n_grains = 240, 45
seeds = rng.random((n_grains, 2)) * size
gy, gx = np.mgrid[0:size, 0:size]
pixels = np.column_stack([gx.ravel(), gy.ravel()])
_, nearest = cKDTree(seeds).query(pixels)
grains = nearest.reshape(size, size)

plt.imshow(grains, cmap='tab20'); plt.title('synthetic grain structure'); plt.axis('off'); plt.show()
print('결정립 %d개' % n_grains)

## 2. 결정립 경계 검출과 크기 분포

In [ ]:
boundary = (ndimage.maximum_filter(grains, 3) != ndimage.minimum_filter(grains, 3))
areas = np.array([int((grains == label).sum()) for label in range(n_grains)])
equivalent_diameter = 2 * np.sqrt(areas / np.pi)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].imshow(boundary, cmap='gray_r'); axes[0].set_title('detected boundaries'); axes[0].axis('off')
axes[1].hist(equivalent_diameter, bins=15)
axes[1].set_xlabel('equivalent grain diameter (pixels)'); axes[1].set_ylabel('count')
plt.tight_layout(); plt.show()
print('평균 결정립 지름 %.1f px, 표준편차 %.1f px' % (equivalent_diameter.mean(), equivalent_diameter.std()))
print('경계 픽셀 비율 %.1f%%' % (100 * boundary.mean()))

## 3. 잡음이 있는 이미지에서 세기

실제 현미경 이미지처럼 잡음을 넣고, 문턱값과 라벨링으로 결정립을 다시 셉니다.

In [ ]:
image = np.where(boundary, 0.2, 0.8) + rng.normal(0, 0.18, (size, size))
smoothed = ndimage.gaussian_filter(image, 1.2)
interior = smoothed > 0.5
labels, count = ndimage.label(interior)
sizes = ndimage.sum(interior, labels, range(1, count + 1))
kept = int((sizes > 30).sum())

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for axis, (title, picture, cmap) in zip(axes, [('noisy image', image, 'gray'),
                                               ('after smoothing', smoothed, 'gray'),
                                               ('labelled grains', labels, 'tab20')]):
    axis.imshow(picture, cmap=cmap); axis.set_title(title); axis.axis('off')
plt.tight_layout(); plt.show()
print('참 결정립 %d개 / 검출 %d개 (30픽셀 이상)' % (n_grains, kept))
print('전처리 설정 하나로 개수가 바뀝니다. 그래서 결정립 계수는 절차를 함께 보고해야 합니다.')
print('최근에는 U-Net 같은 분할 신경망으로 경계를 검출합니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#microstructure)을 여세요.